# PoinTr environment setup (Colab) — clean & ordered

Sets up the **colored PoinTr** fork from scratch on a fresh Colab **GPU** runtime:
clone → deps → CUDA extensions → `pointnet2_ops` → checkpoint → build the color-head model.

**Kullanım:** Runtime = GPU (Runtime ▸ Change runtime type ▸ GPU). Bu notebook'u **baştan sona**
çalıştır. Bittiğinde `model_c` (renkli PoinTr, ağırlıklar yüklü) hazır olur — data/eğitim
hücrelerini AYNI runtime'da bunun altına ekle (Colab her notebook'u ayrı runtime'da açar,
o yüzden hepsi tek notebook / tek oturumda olmalı).

Deps için bilinen düzeltmeler: `requirements.txt`'teki `open3d==0.9` / `timm==0.4.5` py3.11'de
derlenmez → güncelleri kurulur; `numpy<2` sabitlenir; `pointnet2_ops` ayrı kurulur;
`CUDA_HOME` sabit 11.8 yerine otomatik tespit edilir.

In [ ]:
!nvidia-smi -L
# GPU görünmüyorsa: Runtime > Change runtime type > Hardware accelerator = GPU

### 1) Python bağımlılıkları (numpy<2 sabit; open3d/timm güncel)

In [ ]:
# torch 2.11 numpy 2.x'e karşı derli -> numpy'yi DOWNGRADE ETME (mixed-install -> mtrand ABI hatası).
# Pinli eski open3d==0.9 / timm==0.4.5 py3.12'de derlenmez; güncelleri kurulur.
!pip install -q easydict h5py matplotlib opencv-python pyyaml scipy \
    tensorboardX tqdm transforms3d einops timm open3d gdown
import numpy as np; print("deps OK | numpy", np.__version__, "(2.x olmalı)")

> ⚠️ **numpy'yi 2.x'te bırak** (torch 2.11 onu ister). Eğer `numpy.dtype size changed` /
> `mtrand` hatası alırsan numpy karışmış demektir → şunu çalıştır ve **Runtime ▸ Restart**:
> `!pip install --force-reinstall --no-cache-dir "numpy==2.0.2"`  — sonra baştan çalıştır.

### 2) Fork'u klonla → `/content/PoinTr`

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/PoinTr"):
    !git clone https://github.com/eylulpelinkilic/Pelin_Efe_PoinTr.git /content/PoinTr
%cd /content/PoinTr
!git rev-parse --short HEAD

### 3) CUDA build ortamı (sabit değil — otomatik tespit + doğru GPU arch)

In [ ]:
import os, glob, torch
# Colab'da aktif toolkit /usr/local/cuda sembolik linkidir; sürümü hardcode ETME
cuda_home = "/usr/local/cuda" if os.path.isdir("/usr/local/cuda") else sorted(glob.glob("/usr/local/cuda*"))[-1]
os.environ["CUDA_HOME"] = cuda_home
os.environ["PATH"] = f"{cuda_home}/bin:" + os.environ["PATH"]
# eklentiler DOĞRU GPU mimarisi için derlensin (T4=7.5, V100=7.0, A100=8.0, L4=8.9) -> "no kernel image" hatasını önler
cap = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
print("torch", torch.__version__, "| torch-cuda", torch.version.cuda,
      "| CUDA_HOME", cuda_home, "| arch", os.environ["TORCH_CUDA_ARCH_LIST"])
!nvcc --version | tail -2

### 4) `pointnet2_ops` (saf-PyTorch shim — derleme yok)
PoinTr'ın `fps`/`three_nn` gibi ops'ları buna bağlı. Çok yeni torch'ta eski CUDA
repo'su derlenmediği için, kullanılan 6 fonksiyonu saf torch'la enjekte ediyoruz.

In [ ]:
# pointnet2_ops'u DERLEMEK yerine saf-PyTorch SHIM olarak enjekte ediyoruz.
# torch 2.11+cu128 gibi çok yeni stack'te eski CUDA repo'su derlenmiyor. PoinTr sadece
# şu 6 fonksiyonu kullanıyor; hepsi saf torch'la doğru (yerelde brute-force'a karşı test edildi).
# FPS saf-torch döngüsü biraz yavaş ama bu ölçekte (~8k nokta) sorun değil.
import sys, types, torch

def furthest_point_sample(xyz, npoint):        # xyz (B,N,3) -> idx (B,npoint) int32
    B, N, _ = xyz.shape; dev = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, device=dev, dtype=xyz.dtype)
    far = torch.zeros(B, dtype=torch.long, device=dev); ar = torch.arange(B, device=dev)
    for i in range(npoint):
        idx[:, i] = far
        dist = torch.minimum(dist, ((xyz - xyz[ar, far].unsqueeze(1)) ** 2).sum(-1))
        far = torch.max(dist, dim=1).indices
    return idx.int()

def gather_operation(features, idx):           # (B,C,N),(B,S) -> (B,C,S)
    B, C, N = features.shape; idx = idx.long()
    return torch.gather(features, 2, idx.unsqueeze(1).expand(B, C, idx.shape[1])).contiguous()

def three_nn(query, ref):                      # (B,N,3),(B,M,3) -> dist(B,N,3) öklid, idx(B,N,3)
    d = torch.cdist(query, ref)
    dist, idx = torch.topk(d, 3, dim=-1, largest=False, sorted=True)
    return dist.contiguous(), idx.int().contiguous()

def three_interpolate(features, idx, weight):  # (B,C,M),(B,N,3),(B,N,3) -> (B,C,N)
    B, C, M = features.shape; N = idx.shape[1]; idx = idx.long()
    g = torch.gather(features, 2, idx.reshape(B,1,N*3).expand(B,C,N*3)).reshape(B,C,N,3)
    return (g * weight.unsqueeze(1)).sum(-1).contiguous()

def grouping_operation(features, idx):         # (B,C,N),(B,S,K) -> (B,C,S,K)  (SnowFlakeNet için)
    B, C, N = features.shape; _, S, K = idx.shape; idx = idx.long()
    return torch.gather(features, 2, idx.reshape(B,1,S*K).expand(B,C,S*K)).reshape(B,C,S,K).contiguous()

def ball_query(radius, nsample, xyz, new_xyz): # (r,k,(B,N,3),(B,S,3)) -> idx(B,S,k)  (SnowFlakeNet için)
    B, N, _ = xyz.shape; S = new_xyz.shape[1]
    d = torch.cdist(new_xyz, xyz)
    idx = torch.arange(N, device=xyz.device).view(1,1,N).expand(B,S,N).contiguous()
    idx[d > radius] = N
    idx = idx.sort(dim=-1).values[:, :, :nsample]
    first = idx[:, :, 0:1].clone(); first[first == N] = 0
    idx = torch.where(idx == N, first.expand(-1,-1,nsample), idx)
    return idx.int()

_u = types.ModuleType("pointnet2_ops.pointnet2_utils")
for _f in [furthest_point_sample, gather_operation, three_nn, three_interpolate,
           grouping_operation, ball_query]:
    setattr(_u, _f.__name__, _f)
_p = types.ModuleType("pointnet2_ops"); _p.pointnet2_utils = _u
sys.modules["pointnet2_ops"] = _p
sys.modules["pointnet2_ops.pointnet2_utils"] = _u
from pointnet2_ops import pointnet2_utils
print("pointnet2_ops shim enjekte edildi:",
      [n for n in dir(pointnet2_utils) if not n.startswith("_")])

### 5) CUDA extension'ları derle (`chamfer` zorunlu; gridding/cubic GRNet için)

In [ ]:
import subprocess
EXTS = ["chamfer_dist", "gridding", "gridding_loss", "cubic_feature_sampling"]  # emd PoinTr için gerekmez
for ext in EXTS:
    print(f"── building {ext} ──")
    # --no-build-isolation: bu setup.py'ler de torch.utils.cpp_extension'a bağlı
    r = subprocess.run("pip install -q --no-build-isolation .", shell=True,
                       cwd=f"/content/PoinTr/extensions/{ext}", capture_output=True, text=True)
    ok = r.returncode == 0
    print("   ", "✅ ok" if ok else "‼ FAILED")
    if not ok:
        print(r.stdout[-600:]); print(r.stderr[-1800:])

### 6) Her şey import oluyor mu? (GPU smoke test)

In [ ]:
import os, sys
sys.path.insert(0, "/content/PoinTr"); os.chdir("/content/PoinTr")
import torch, numpy as np
print("numpy", np.__version__, "| torch", torch.__version__)
import chamfer, gridding, gridding_distance, cubic_feature_sampling
from pointnet2_ops import pointnet2_utils
from extensions.chamfer_dist import ChamferDistanceL1
from models.PoinTr import PoinTr, Fold, fps
from models.dgcnn_group import DGCNN_Grouper
from models.Transformer import PCTransformer
# gerçekten GPU'da çalışıyor mu: fps + chamfer
x = torch.rand(1, 1024, 3, device="cuda")
idx = pointnet2_utils.furthest_point_sample(x, 128)
d = ChamferDistanceL1()(x, torch.rand(1, 512, 3, device="cuda"))
print("pointnet2 fps:", tuple(idx.shape), "| chamfer:", float(d))
print("✅ PoinTr environment READY")

### 7) Pretrained checkpoint (ShapeNet55)

In [ ]:
import os, subprocess
CKPT = "/content/PoinTr/ckpts/PoinTr_ShapeNet55.pth"
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 50e6:
    subprocess.run(f"gdown 1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ -O {CKPT}", shell=True, check=True)
print("checkpoint MB:", round(os.path.getsize(CKPT)/1e6, 1), " (>400 olmalı)")

### 8) Renkli PoinTr modelini kur (senin color-head değişikliğin)
Pretrained state_dict → `base`, cihaz → `DEV`, sonra 6D grouper + `color_head` monkey-patch.

In [ ]:
import torch
DEV = "cuda"
sd = torch.load(CKPT, map_location="cpu")
key = "base_model" if "base_model" in sd else ("model" if "model" in sd else None)
base = sd[key] if key else sd
base = {k.replace("module.", ""): v for k, v in base.items()}
print("state_dict anahtar sayısı:", len(base))

In [ ]:
import torch, torch.nn as nn
from easydict import EasyDict
from models.dgcnn_group import DGCNN_Grouper
from models.Transformer import PCTransformer
from models.PoinTr import PoinTr, Fold, fps

# --- grouper: orijinalin BIREBIR kopyasi (input_trans 6D), sarmalama yok ---
def grouper_init(self):
    nn.Module.__init__(self)
    self.input_trans = nn.Conv1d(6, 8, 1)
    self.layer1 = nn.Sequential(nn.Conv2d(16,32,1,bias=False),  nn.GroupNorm(4,32),  nn.LeakyReLU(0.2))
    self.layer2 = nn.Sequential(nn.Conv2d(64,64,1,bias=False),  nn.GroupNorm(4,64),  nn.LeakyReLU(0.2))
    self.layer3 = nn.Sequential(nn.Conv2d(128,64,1,bias=False), nn.GroupNorm(4,64),  nn.LeakyReLU(0.2))
    self.layer4 = nn.Sequential(nn.Conv2d(128,128,1,bias=False),nn.GroupNorm(4,128), nn.LeakyReLU(0.2))
def grouper_forward(self, x):
    coor = x[:, :3].contiguous(); f = self.input_trans(x)
    f = self.get_graph_feature(coor,f,coor,f); f=self.layer1(f); f=f.max(-1)[0]
    cq,fq = self.fps_downsample(coor,f,512); f=self.get_graph_feature(cq,fq,coor,f); f=self.layer2(f); f=f.max(-1)[0]; coor=cq
    f = self.get_graph_feature(coor,f,coor,f); f=self.layer3(f); f=f.max(-1)[0]
    cq,fq = self.fps_downsample(coor,f,128); f=self.get_graph_feature(cq,fq,coor,f); f=self.layer4(f); f=f.max(-1)[0]; coor=cq
    return coor, f
DGCNN_Grouper.__init__ = grouper_init
DGCNN_Grouper.forward  = grouper_forward

# --- PoinTr: orijinalin BIREBIR kopyasi + color_head, sarmalama yok ---
def poinTr_init(self, config, **kw):
    nn.Module.__init__(self)
    self.trans_dim=config.trans_dim; self.knn_layer=config.knn_layer
    self.num_pred=config.num_pred;  self.num_query=config.num_query
    self.fold_step=int(pow(self.num_pred//self.num_query,0.5)+0.5)
    self.base_model=PCTransformer(in_chans=3, embed_dim=self.trans_dim, depth=[6,8], drop_rate=0., num_query=self.num_query, knn_layer=self.knn_layer)
    self.foldingnet=Fold(self.trans_dim, step=self.fold_step, hidden_dim=256)
    self.color_head=Fold(self.trans_dim, step=self.fold_step, hidden_dim=256)
    self.increase_dim=nn.Sequential(nn.Conv1d(self.trans_dim,1024,1), nn.BatchNorm1d(1024), nn.LeakyReLU(0.2), nn.Conv1d(1024,1024,1))
    self.reduce_map=nn.Linear(self.trans_dim+1027, self.trans_dim)
def poinTr_forward(self, xyz):
    q, coarse = self.base_model(xyz); B,M,C = q.shape
    gf = self.increase_dim(q.transpose(1,2)).transpose(1,2); gf = torch.max(gf,1)[0]
    rf = self.reduce_map(torch.cat([gf.unsqueeze(-2).expand(-1,M,-1), q, coarse],-1).reshape(B*M,-1))
    reb_xyz = (self.foldingnet(rf).reshape(B,M,3,-1) + coarse.unsqueeze(-1)).transpose(2,3).reshape(B,-1,3)
    reb_rgb = torch.sigmoid(self.color_head(rf)).reshape(B,M,3,-1).transpose(2,3).reshape(B,-1,3)
    reb = torch.cat([reb_xyz, reb_rgb], -1)
    inp = fps(xyz[:,:,:3].contiguous(), self.num_query)
    return torch.cat([coarse, inp],1), torch.cat([reb, xyz],1)
PoinTr.__init__ = poinTr_init
PoinTr.forward  = poinTr_forward

cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
model_c = PoinTr(cfg)
for k in ["base_model.grouper.input_trans.weight","base_model.grouper.input_trans.bias"]:
    base.pop(k, None)   # 3->6 kanal, şekil değişti: yükleme
mi, ui = model_c.load_state_dict(base, strict=False)
print(f"OK | missing={len(mi)} unexpected={len(ui)}  (missing=color_head.* + input_trans.*, unexpected~0 beklenir)")
model_c.to(DEV)
print("✅ renkli PoinTr (model_c) hazır")

---
## Data + end-to-end training (colored PoinTr `model_c`)

Buradan sonrası `model_c` üzerine kurulu (yukarıda ağırlıklar yüklendi). `model_c` **6D partial**
(xyzrgb) alır, o yüzden ayrı bir frozen orijinal PoinTr'a gerek yok. Akış: renkli GT PLY'leri
yükle → occlude → cache → `model_c`'yi chamfer+renk kaybıyla eğit.

> **Not:** aşağıdaki eğitim birkaç bulutu **ezberleten** bir sanity-check (color_head gerçekten
> renk öğrenebiliyor mu?). Gerçek eğitim için `subset`'i tüm `DATA`'ya çıkar, train/val ayır,
> step sayısını artır ve val ΔE'sine bak.

### A) Yardımcılar + **hazır occluded partial'ları** yükle (occluded.ipynb çıktısı)

In [ ]:
import numpy as np, glob, random, zipfile, os, open3d as o3d
from scipy.spatial import cKDTree

def load_ply(p, n=8192):
    pc = o3d.io.read_point_cloud(p)
    if len(pc.points) > n: pc = pc.farthest_point_down_sample(n)   # uniform FPS -> n
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if len(rgb) != len(xyz): rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], 1).astype(np.float32)

def pc_norm(gt):                                    # unit-sphere (PoinTr eğitim konvansiyonu)
    x = gt[:, :3] - gt[:, :3].mean(0)
    x = x / (np.linalg.norm(x, axis=1).max() + 1e-9)
    return np.concatenate([x, gt[:, 3:6]], 1).astype(np.float32)

def separate_colored(gt, crop=0.5, seed=0):         # -> partial(6D), mask(True=eksik)
    xyz = gt[:, :3]; N = len(gt); nc = int(round(N * crop))
    c = xyz.mean(0); xyzn = (xyz - c) / (np.linalg.norm(xyz - c, axis=1).max() + 1e-9)
    rng = np.random.default_rng(seed); v = rng.standard_normal(3); v /= np.linalg.norm(v)
    order = np.argsort(np.linalg.norm(xyzn - v[None], axis=1))
    mask = np.zeros(N, bool); mask[order[:nc]] = True
    return gt[order[nc:]], mask

def srgb_to_lab(rgb):
    rgb = np.clip(rgb, 0, 1); lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124,0.3576,0.1805],[0.2126,0.7152,0.0722],[0.0193,0.1192,0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047, 1.0, 1.08883]); d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    return np.stack([116*f[:,1]-16, 500*(f[:,0]-f[:,1]), 200*(f[:,1]-f[:,2])], 1)
def deltaE(a, b): return np.linalg.norm(srgb_to_lab(a) - srgb_to_lab(b), axis=1)
print("yardımcılar hazır")

In [ ]:
from google.colab import files
os.makedirs("/content/occluded", exist_ok=True)
up = files.upload()                                 # occluded_occ.zip seç (occluded.ipynb çıktısı)
with zipfile.ZipFile(list(up.keys())[0]) as f: f.extractall("/content/occluded")
n = len([p for p in glob.glob("/content/occluded/**/*.ply", recursive=True)
         if not p.endswith("_missing.ply")])
print(n, "partial PLY bulundu (occluded)")

### B) Cache: her partial + `_missing` → tam GT = partial ∪ missing
Yeniden occlude ETMİYORUZ — senin resmi partial'larını kullanıyoruz; GT'yi birleşimden kurup
tam-bulut çerçevesinde unit-sphere normalize ediyoruz (partial ve GT aynı çerçevede).

In [ ]:
DIFFICULTY, NCLOUDS = "moderate", 50     # simple=25% / moderate=50% / hard=75%
N_IN = 8192                              # partial -> PoinTr girişi (FPS)

def load_fps(ply, n):
    pc = o3d.io.read_point_cloud(ply)
    if len(pc.points) > n: pc = pc.farthest_point_down_sample(n)
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if len(rgb) != len(xyz): rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], 1).astype(np.float32)

# hazır partial'ları bul (senin occluded.ipynb çıktın), *_missing hariç
parts = sorted(p for p in glob.glob(f"/content/occluded/**/{DIFFICULTY}/**/*.ply", recursive=True)
               if not p.endswith("_missing.ply"))
if not parts:  # difficulty alt-klasörü farklı konumdaysa gevşek ara
    parts = sorted(p for p in glob.glob("/content/occluded/**/*.ply", recursive=True)
                   if f"/{DIFFICULTY}/" in p and not p.endswith("_missing.ply"))
assert parts, f"{DIFFICULTY} altında partial PLY yok — zip yapısını/difficulty adını kontrol et"
random.Random(0).shuffle(parts); parts = parts[:NCLOUDS]

DATA = []
for j, p in enumerate(parts):
    mp = p[:-4] + "_missing.ply"
    if not os.path.exists(mp):
        continue                          # missing yoksa GT kuramayız (SAVE_MISSING=True olmalı)
    partial_raw = load_fps(p, N_IN)
    missing_raw = load_fps(mp, N_IN)
    gt_raw = np.vstack([partial_raw, missing_raw])          # tam renkli GT = partial ∪ missing
    c = gt_raw[:, :3].mean(0); s = np.linalg.norm(gt_raw[:, :3] - c, axis=1).max() + 1e-9
    nrm = lambda a: np.concatenate([(a[:, :3] - c) / s, a[:, 3:6]], 1).astype(np.float32)
    partial, gt = nrm(partial_raw), nrm(gt_raw)             # ikisi de TAM-bulut çerçevesinde (unit sphere)
    miss = np.zeros(len(gt), bool); miss[len(partial_raw):] = True
    DATA.append(dict(gt=gt, partial=partial, miss=miss))
    if j % 10 == 0: print(f"  {j+1}/{len(parts)}  partial={len(partial)}  missing={miss.sum()}")
assert DATA, "hiç (partial, missing) çifti yüklenemedi"
print(f"DATA hazır: {len(DATA)} bulut  (difficulty={DIFFICULTY})")

---
## PART-BASED colored completion (dondurulmuş geometri + part-tutarlı renk)

Senin fikrin: parçalar renk-tutarlı → tamamlanan noktayı en yakın görünürün gürültülü rengiyle
değil, **kendi parçasının görünür-ortalama rengiyle** boya.

* **geometri:** DONDURULMUŞ orijinal PoinTr (pretrained, bozulmaz) — **asıl kazanç bu**,
  uçtan-uca eğitim faciasının tersi. Gerçek uçak geometrisi verir.
* **renklendirme:** `part_color` şimdilik rgb-kümeleme **placeholder**'ı (denetimsiz → NN'i zar zor
  geçer). Fikrin GERÇEK kazancı (toy'da oracle 2.6×) `part_color_oracle` ile, yani **gerçek parça
  etiketiyle** gelir → sıradaki iş: ShapeNet-Part etiketlerini `scripts/align_partseg.py` ile GT'ye
  kaynaştırmak, sonra eğitilmiş PointNet part-seg.

> ⚠️ **Sıra:** env(1–7) → checkpoint → **model_c (monkey-patch) hücrelerini ATLA** → upload/PLY →
> cache → bu hücreler. model_c orijinal PoinTr sınıfını 6D'ye çevirir; çalıştırdıysan Restart edip atla
> (P1 assert doğrular).

### P1) Dondurulmuş ORİJİNAL PoinTr (geometri)

In [ ]:
import torch
from easydict import EasyDict
from models.PoinTr import PoinTr, fps

DEV = "cuda"
cfg = EasyDict(trans_dim=384, knn_layer=1, num_pred=6144, num_query=96)
geo = PoinTr(cfg)
sd = torch.load(CKPT, map_location="cpu")
base = sd.get("base_model", sd.get("model", sd))
base = {k.replace("module.", ""): v for k, v in base.items()}
mi, ui = geo.load_state_dict(base, strict=False)
assert len(mi) == 0, (f"ORİJİNAL PoinTr bekleniyordu ama missing={mi[:4]}... "
                      "model_c (monkey-patch) hücrelerini atlamadın. Runtime>Restart, model_c'yi atla.")
for p in geo.parameters(): p.requires_grad_(False)
geo.eval().to(DEV)
print(f"dondurulmuş orijinal PoinTr hazır | num_pred={geo.num_pred} | missing=0")

### P2) Yardımcılar: geometri tamamla + part-tutarlı renklendir

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans

@torch.no_grad()
def complete_geometry(partial):
    """partial (N,6) -> tamamlanan YENİ noktalar (num_pred,3)."""
    p = torch.from_numpy(partial[:, :3]).float().unsqueeze(0).to(DEV)
    fine = geo(p)[1][0, :geo.num_pred]          # ilk num_pred = üretilen yeni noktalar
    return fine.cpu().numpy()

def nn_color(partial, comp):                    # BASELINE: en yakın görünür noktanın rengi
    _, i = cKDTree(partial[:, :3]).query(comp[:, :3], k=1)
    return partial[i, 3:6]

def part_color(partial, comp, K=6, seed=0):
    """PLACEHOLDER part-seg: görünür renkleri K kümeye ayır (parça renk-tutarlı varsayımı),
       tamamlananı en yakın görünür noktanın kümesine ata, o kümenin merkez rengiyle boya.
       NOT: bu DENETİMSİZ proxy NN'i zar zor geçer (toy: 27 vs 28). GERÇEK kazanç
       (toy oracle: 11, ~2.6x) ancak GERÇEK parça etiketiyle gelir -> ShapeNet-Part
       fusion (scripts/align_partseg.py) veya eğitilmiş PointNet part-seg."""
    vx, vr = partial[:, :3], partial[:, 3:6]
    km = KMeans(K, n_init=4, random_state=seed).fit(vr)
    _, i = cKDTree(vx).query(comp[:, :3], k=1)   # tamamlananı en yakın görünüre bağla
    return km.cluster_centers_[km.labels_[i]], 0

def part_color_oracle(partial, comp, gt_xyz, gt_lab):
    """GERÇEK method: tamamlanan nokta -> en yakın GT noktasının parça etiketi (oracle seg),
       renk = o parçanın GÖRÜNÜR-ortalama rengi. gt_lab = ShapeNet-Part etiketleri (henüz yok)."""
    vx, vr = partial[:, :3], partial[:, 3:6]
    _, gi = cKDTree(gt_xyz).query(comp[:, :3], k=1); comp_lab = gt_lab[gi]
    _, vi = cKDTree(gt_xyz).query(vx, k=1); vis_lab = gt_lab[vi]
    P = int(gt_lab.max()) + 1; glob = vr.mean(0); mean = np.tile(glob, (P, 1))
    for k in range(P):
        m = vis_lab == k
        if m.any(): mean[k] = vr[m].mean(0)
    return mean[comp_lab]
print("part-coloring yardımcıları hazır (part_color=placeholder, part_color_oracle=etiket gerektirir)")

### P3) Tüm bulutlarda ΔE: part-based vs NN-kopya (eksik bölge)

In [ ]:
K = 6
nn_all, pc_all, fb = [], [], 0
for d in DATA[:20]:
    gt, partial = d["gt"], d["partial"]
    comp = complete_geometry(partial)
    _, gi = cKDTree(gt[:, :3]).query(comp[:, :3], k=1)   # eval referansı: en yakın GT
    true_rgb = gt[gi, 3:6]; is_miss = d["miss"][gi]      # eksik-bölge maskesi
    nn_rgb = nn_color(partial, comp)
    pc_rgb, absent = part_color(partial, comp, K=K); fb += absent
    m = is_miss if is_miss.any() else np.ones(len(comp), bool)
    nn_all.append(deltaE(nn_rgb[m], true_rgb[m]).mean())
    pc_all.append(deltaE(pc_rgb[m], true_rgb[m]).mean())
nn_all, pc_all = np.array(nn_all), np.array(pc_all)
print(f"eksik-bölge ortalama ΔE(Lab)  ({len(nn_all)} bulut, K={K} parça):")
print(f"  NN-kopya (baseline) : {nn_all.mean():6.3f} ± {nn_all.std():.2f}")
print(f"  part-based (placeh.): {pc_all.mean():6.3f} ± {pc_all.std():.2f}")
print(f"  part-based kazandığı bulut sayısı: {(pc_all < nn_all).sum()}/{len(nn_all)}")
print(f"  (tamamen kapalı parça -> global-mean fallback olayı: {fb})")

> **Okuma:** placeholder (rgb-kümeleme) ≈ NN çıkarsa bu beklenen — denetimsiz kümeleme eksik bölgeyi
> semantik parçalara bölemez. Fikrin çalıştığını görmek için `part_color`'ı `part_color_oracle` ile
> değiştir (gerçek parça etiketi gerekir). Geometri panelinin düzgün uçak vermesi = dondurulmuş
> PoinTr'ın iş gördüğü, faciayı geçtiğimiz anlamına gelir.

### P4) Görsel: occluded / NN-kopya / part-based / GT

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
d = DATA[0]; gt, partial = d["gt"], d["partial"]
comp = complete_geometry(partial)
nn_rgb = nn_color(partial, comp); pc_rgb, _ = part_color(partial, comp, K=6)
full_nn = np.vstack([partial, np.hstack([comp, nn_rgb])])
full_pc = np.vstack([partial, np.hstack([comp, pc_rgb])])
def tr(a):
    c = ["rgb(%d,%d,%d)" % (int(r*255), int(g*255), int(b*255)) for r, g, b in np.clip(a[:,3:6],0,1)]
    return go.Scatter3d(x=a[:,0], y=a[:,1], z=a[:,2], mode="markers", marker=dict(size=1.6, color=c))
panels = [("occluded (input)", partial), ("NN-kopya", full_nn),
          ("part-based", full_pc), ("GT", gt)]
fig = make_subplots(rows=1, cols=4, specs=[[{"type":"scene"}]*4], subplot_titles=[t for t,_ in panels])
for i,(_,a) in enumerate(panels,1): fig.add_trace(tr(a),1,i)
for s in ("scene","scene2","scene3","scene4"): fig.layout[s].aspectmode="data"
fig.update_layout(height=520, showlegend=False, margin=dict(l=0,r=0,t=30,b=0)); fig.show()